# Phase 3.2: Spectator Flow Agent (MQTT Publisher)

This notebook runs Section A4 halftime simulation and publishes spectator and movement state payloads to MQTT.

In [11]:
# Cell 2 purpose: Import project modules, MQTT helpers, and payload builders for Phase 3.2 publishing.
import math
import statistics
import sys
import uuid
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))

from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.mqtt_payloads import (
    build_movement_state_payload,
    build_spectator_event_payload,
    validate_movement_state_payload,
    validate_spectator_event_payload,
 )
from simulated_city.simulation_core import simulate_halftime_from_app_config
from simulated_city.topic_schema import topic_movement_state, topic_spectator_events

In [12]:
# Cell 3 purpose: Load typed config and print publishing controls from halftime + halftime_map.
from types import SimpleNamespace

app_config = load_config()
if app_config.halftime is None:
    raise ValueError('Missing halftime section in config.yaml')

halftime_cfg = app_config.halftime
halftime_map_cfg = getattr(app_config, 'halftime_map', None)
if halftime_map_cfg is None:
    halftime_map_cfg = SimpleNamespace(
        publish_interval_s=1,
        max_points_per_message=1000,
        zone_naming=SimpleNamespace(
            canonical_service_zones=('zone_1', 'zone_2'),
            legacy_zone_aliases={'zone_a': 'zone_1', 'zone_b': 'zone_2'},
        ),
        zone_1_toilet_w=SimpleNamespace(lng=12.5678, lat=55.6762),
        zone_1_toilet_m=SimpleNamespace(lng=12.5679, lat=55.67622),
        zone_1_cafe=SimpleNamespace(lng=12.5680, lat=55.67618),
        zone_2_toilet_w=SimpleNamespace(lng=12.5689, lat=55.6760),
        zone_2_toilet_m=SimpleNamespace(lng=12.5690, lat=55.67602),
        zone_2_cafe=SimpleNamespace(lng=12.5691, lat=55.67598),
    )

params = halftime_cfg.to_simulation_core_kwargs()

run_id = f'a4-run-{uuid.uuid4().hex[:8]}'
seed_mode = 'random each run (seed=null)' if halftime_cfg.seed is None else f'fixed seed={halftime_cfg.seed}'

print('Loaded halftime parameters from config.yaml (Phase 3.2 publisher mode):')
print(f'  run_id: {run_id}')
print(f'  seed_mode: {seed_mode}')
for key, value in params.items():
    print(f'  {key}: {value}')

print('\nLoaded halftime_map publish controls:')
print(f'  publish_interval_s: {halftime_map_cfg.publish_interval_s}')
print(f'  max_points_per_message: {halftime_map_cfg.max_points_per_message}')
print(f'  canonical_service_zones: {halftime_map_cfg.zone_naming.canonical_service_zones}')
print(f'  legacy_zone_aliases: {halftime_map_cfg.zone_naming.legacy_zone_aliases}')

Loaded halftime parameters from config.yaml (Phase 3.2 publisher mode):
  run_id: a4-run-cd128e3a
  seed_mode: random each run (seed=null)
  seed: None
  spectator_count: 1000
  halftime_duration_s: 900
  toilet_servers: 24
  cafe_servers: 16
  toilet_service_min_s: 45
  toilet_service_max_s: 220
  cafe_service_min_s: 20
  cafe_service_max_s: 90
  urinal_service_min_s: 15
  urinal_service_max_s: 70
  inter_facility_walk_s: 30
  seat_leave_rate: 0.7
  women_ratio: 0.3
  shared_urinal_total: 16

Loaded halftime_map publish controls:
  publish_interval_s: 1
  max_points_per_message: 1000
  canonical_service_zones: ('zone_1', 'zone_2')
  legacy_zone_aliases: {'zone_a': 'zone_1', 'zone_b': 'zone_2'}


In [13]:
# Cell 4 purpose: Run simulation from config and print baseline KPI metrics before publishing.
print('Running simulation from config...')
result = simulate_halftime_from_app_config(app_config)
print(f'Simulation complete: {len(result.ticks)} ticks collected')

print('\n=== Key Performance Indicators (KPIs) ===')
print(f'Max Queue Length: {result.max_queue_length}')
print(f'Average Wait Time (overall): {result.average_wait_s:.2f} seconds')
print(f'Average Wait Time (toilet queues): {result.average_wait_toilet_s:.2f} seconds')
print(f'Average Wait Time (cafe queues): {result.average_wait_cafe_s:.2f} seconds')
print(f'Average Wait Time (urinal queue): {result.average_wait_urinal_s:.2f} seconds')
print(f'Missed Kickoff Count: {result.missed_kickoff_count}')
print(f'Made Kickoff Count: {result.made_kickoff_count}')
print(f'Stayed Seated During Halftime: {result.stayed_seated_count}')
print(f'Went Down During Halftime: {result.went_down_count}')
print(f'Went Down and Made It Back: {result.went_down_made_back_count}')
print(f'Total Served Tasks: {result.total_served_tasks}')

Running simulation from config...
Simulation complete: 901 ticks collected

=== Key Performance Indicators (KPIs) ===
Max Queue Length: 147
Average Wait Time (overall): 116.60 seconds
Average Wait Time (toilet queues): 145.46 seconds
Average Wait Time (cafe queues): 170.07 seconds
Average Wait Time (urinal queue): 0.16 seconds
Missed Kickoff Count: 436
Made Kickoff Count: 564
Stayed Seated During Halftime: 346
Went Down During Halftime: 654
Went Down and Made It Back: 218
Total Served Tasks: 497


In [14]:
# Cell 5 purpose: Connect to MQTT and prepare topics/schema for spectator + movement publishing.
schema_version = '1.0'
spectator_topic = topic_spectator_events()
movement_topic = topic_movement_state()

mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='spectator-flow')
print(f'Connected to MQTT broker at {app_config.mqtt.host}:{app_config.mqtt.port}')
print(f'run_id: {run_id}, schema_version: {schema_version}')
print(f'spectator_topic: {spectator_topic}')
print(f'movement_topic: {movement_topic}')
print(f'publish_interval_s: {halftime_map_cfg.publish_interval_s}')
print(f'max_points_per_message: {halftime_map_cfg.max_points_per_message}')

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Connected to MQTT broker at 127.0.0.1:1883
run_id: a4-run-cd128e3a, schema_version: 1.0
spectator_topic: stadium/a4/halftime/events/spectator
movement_topic: stadium/a4/halftime/state/movement
publish_interval_s: 1
max_points_per_message: 1000


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


In [ ]:
# Cell 6 purpose: Publish spectator events and movement snapshots with shared run_id, throttle, and payload cap.
publish_every_s = halftime_map_cfg.publish_interval_s
max_points = halftime_map_cfg.max_points_per_message

published_spectator_count = 0
published_movement_count = 0
last_movement_payload = None

zone_targets = [
    ('zone_1', 'toilet_w', halftime_map_cfg.zone_1_toilet_w),
    ('zone_1', 'toilet_m', halftime_map_cfg.zone_1_toilet_m),
    ('zone_1', 'cafe', halftime_map_cfg.zone_1_cafe),
    ('zone_2', 'toilet_w', halftime_map_cfg.zone_2_toilet_w),
    ('zone_2', 'toilet_m', halftime_map_cfg.zone_2_toilet_m),
    ('zone_2', 'cafe', halftime_map_cfg.zone_2_cafe),
]

def _publish_with_retry(topic, payload):
    global mqtt_client
    attempts = 0
    while attempts < 2:
        attempts += 1
        try:
            return mqtt.publish_json_checked(mqtt_client, topic, payload, qos=1)
        except RuntimeError:
            connector = getattr(mqtt_client, '_simcity_connector', None)
            if connector is not None:
                connector.disconnect()
            mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='spectator-flow')
    return False

for tick in result.ticks:
    if tick.timestamp_s % publish_every_s != 0:
        continue

    spectator_payload = build_spectator_event_payload(
        schema_version=schema_version,
        run_id=run_id,
        timestamp_s=tick.timestamp_s,
        spectators_out_of_seat=tick.spectators_out_of_seat,
        queue_toilet=tick.queue_lengths['toilet'],
        queue_cafe=tick.queue_lengths['cafe'],
        stayed_seated_count=result.stayed_seated_count,
        went_down_count=result.went_down_count,
        went_down_made_back_count=result.went_down_made_back_count,
    )
    validate_spectator_event_payload(spectator_payload)

    if _publish_with_retry(spectator_topic, spectator_payload):
        published_spectator_count += 1

    movement_points = min(max_points, tick.spectators_out_of_seat)
    spectators = []
    for spectator_id in range(1, movement_points + 1):
        zone_name, task_name, anchor = zone_targets[spectator_id % len(zone_targets)]
        angle = (tick.timestamp_s + spectator_id) * 0.15
        lng = anchor.lng + math.cos(angle) * 0.00003
        lat = anchor.lat + math.sin(angle) * 0.00003
        spectators.append(
            {
                'spectator_id': spectator_id,
                'state': 'WALKING_TO_ZONE' if tick.spectators_out_of_seat > 0 else 'SEATED',
                'target': f'{zone_name}_{task_name}',
                'lng': round(lng, 6),
                'lat': round(lat, 6),
            }
        )

    movement_payload = build_movement_state_payload(
        schema_version=schema_version,
        run_id=run_id,
        timestamp_s=tick.timestamp_s,
        spectators=spectators,
    )
    validate_movement_state_payload(movement_payload)

    if _publish_with_retry(movement_topic, movement_payload):
        published_movement_count += 1
        last_movement_payload = movement_payload

print(f'Published spectator payloads: {published_spectator_count}')
print(f'Published movement payloads: {published_movement_count}')
if last_movement_payload is not None:
    print(f"Last movement payload spectator count: {len(last_movement_payload['spectators'])}")
    print(f"Last movement payload timestamp_s: {last_movement_payload['timestamp_s']}")

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


RuntimeError: Message publish failed: The client is not currently connected.

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

In [ ]:
# Cell 7 purpose: Print queue summary and disconnect MQTT connection cleanly.
toilet_queues = [tick.queue_lengths['toilet'] for tick in result.ticks]
cafe_queues = [tick.queue_lengths['cafe'] for tick in result.ticks]

print('\n=== Queue Evolution Summary ===')
print(f"Toilet Queue - Max: {max(toilet_queues)}, Avg: {statistics.mean(toilet_queues):.1f}, Peak at second: {toilet_queues.index(max(toilet_queues))}")
print(f"Cafe Queue - Max: {max(cafe_queues)}, Avg: {statistics.mean(cafe_queues):.1f}, Peak at second: {cafe_queues.index(max(cafe_queues))}")

connector = getattr(mqtt_client, '_simcity_connector', None)
if connector is not None:
    connector.disconnect()
    print('Disconnected from MQTT broker.')

print('\n=== Phase 3.2 Publisher Complete ===')
print('Spectator and movement payloads were published to MQTT with a shared run_id.')


=== Queue Evolution Summary ===
Toilet Queue - Max: 122, Avg: 63.7, Peak at second: 803
Cafe Queue - Max: 252, Avg: 121.6, Peak at second: 871

=== Phase 1.2 Local Agent Complete ===
No MQTT connection was created. All movement/task outputs stayed in notebook memory.
